# SVM

Implementasi SVM mencakup kernel Linear, Polynomial, dan Gaussian. Evaluasi
membandingkan ketiganya dengan scikit-learn. Visualisasi Matplotlib memperlihatkan
batas keputusan dan margin, sedangkan PSO digunakan untuk mencari hyperparameter.
Model terpilih disimpan untuk menghasilkan prediksi dalam format CSV.

Secara default, evaluasi menggunakan seluruh data latih. Eksperimen bonus memakai
sampel terpisah. Seluruh alur telah diuji pada 320 baris data latih dan 12 baris
data test. Eksekusi pada seluruh dataset belum diverifikasi.

Grafik Matplotlib ditampilkan di notebook dan disimpan sebagai PNG.


Jalankan sel notebook secara berurutan.

Perhitungan kernel menggunakan CUDA jika tersedia, dengan CPU sebagai fallback.
Solver optimasi SciPy tetap berjalan di CPU. Artefak akhir disimpan sebagai
`output/models/svm.pkl` dan `output/submissions/submission_svm.csv`.

In [ ]:
import matplotlib.pyplot as plt
import pickle
from scipy.optimize import minimize
from matplotlib.lines import Line2D
# Siapkan library serta impor kelas SVM dan helper visualisasi.
import hashlib
import inspect
import json
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from IPython.display import Code, Markdown, display
from sklearn.metrics import f1_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.svm import SVC
from threadpoolctl import threadpool_limits

# Cari root repo agar impor tetap bekerja dari folder notebook mana pun.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'data/train.csv').is_file())
# Kelas ini juga dapat diimpor di bagian SVM pada notebook kelompok.


# Seed mempertahankan pembagian fold dan holdout kalibrasi.
SEED = 42
# Gunakan None untuk seluruh data latih, atau isi jumlah baris untuk evaluasi sampel.
CV_ROWS = None
N_FOLDS = 5
DEV_SIZE, HOLDOUT_SIZE, VISUAL_SIZE = 800, 400, 400
# Delapan partikel × empat generasi membatasi pencarian menjadi 32 kandidat.
PARTICLES, GENERATIONS = 8, 4
BASE_PARAMS = dict(kernel='gaussian', tol=0.01, max_iter=1000, seed=SEED, device='auto')
DATA_DIR = ROOT / 'data'
FIG_DIR = ROOT / 'output' / 'figures'
RUN_DIR = ROOT / 'output' / 'experiments' / 'svm'
MODEL_PATH = ROOT / 'output' / 'models' / 'svm.pkl'
SUB_PATH = ROOT / 'output' / 'submissions' / 'submission_svm.csv'
FIG_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
# C menyeimbangkan penalti kesalahan dengan lebar margin. Nilai 0.1 berasal dari eksperimen linear.
# Polynomial derajat 3 menangkap interaksi; coef0=1 mempertahankan orde rendah.
# Gaussian memakai C=1 sebagai baseline. gamma='scale' mengikuti varians train fold.
KERNEL_PARAMS = {
    'linear': dict(kernel='linear', C=0.1),
    'poly': dict(kernel='poly', C=1.0, degree=3, gamma='scale', coef0=1.0),
    'gaussian': dict(kernel='gaussian', C=1.0, gamma='scale'),
}
KERNEL_NAMES = {'linear': 'Linear', 'poly': 'Polynomial', 'gaussian': 'Gaussian'}
# tol=0.01 mengatur toleransi KKT. max_iter membatasi jumlah putaran optimasi blok.
COMMON_PARAMS = dict(tol=0.01, max_iter=1000, seed=SEED, device='auto')
print('NumPy', np.__version__, '| pandas', pd.__version__, '| SciPy', scipy.__version__)

In [ ]:
class SVM:

    # C mengatur penalti margin, tol toleransi optimasi, dan max_iter batas putaran training.
    def __init__(self, C=1.0, tol=1e-3, max_iter=1000, seed=42,
                 kernel='linear', degree=3, gamma='scale', coef0=0.0, device='cpu'):
        self.C = C
        self.tol = tol
        self.max_iter = max_iter
        self.seed = seed
        self.kernel = kernel
        self.degree = degree
        self.gamma = gamma
        self.coef0 = coef0
        self.device = device

    # Latih SVM, lalu pelajari kalibrasi probabilitas dari holdout di dalam data train.
    def fit(self, X, y):
        X = self._validate_X(X)
        y = np.asarray(y)
        if y.ndim != 1 or len(y) != len(X) or not np.isfinite(y).all():
            raise ValueError('y harus berupa vektor label numerik finite sepanjang X.')
        classes = np.unique(y)
        if len(classes) != 2:
            raise ValueError('SVM membutuhkan tepat dua kelas pada data train.')
        if not np.isfinite(self.C) or self.C <= 0:
            raise ValueError('C harus positif dan finite.')
        if not np.isfinite(self.tol) or self.tol <= 0:
            raise ValueError('tol harus positif dan finite.')
        if not isinstance(self.max_iter, (int, np.integer)) or self.max_iter < 1:
            raise ValueError('max_iter harus berupa bilangan bulat positif.')
        # Alias lama dipertahankan agar pemanggil dan model tersimpan tetap dapat digunakan.
        if self.kernel not in ('linear', 'poly', 'gaussian', 'rbf'):
            raise ValueError("kernel harus 'linear', 'poly', atau 'gaussian'.")
        if not isinstance(self.degree, (int, np.integer)) or self.degree < 1:
            raise ValueError('degree harus berupa bilangan bulat positif.')
        if not np.isfinite(self.coef0) or self.coef0 < 0:
            raise ValueError('coef0 harus nonnegatif dan finite.')
        if isinstance(self.gamma, str):
            if self.gamma not in ('scale', 'auto'):
                raise ValueError("gamma harus 'scale', 'auto', atau angka positif.")
        elif not np.isfinite(self.gamma) or self.gamma <= 0:
            raise ValueError('gamma harus positif dan finite.')

        self._fit_svm(X, y)
        self._calibrate(X, y)
        return self

    # Pecah QP dual menjadi blok kecil agar tidak menyimpan matriks kernel n x n.
    def _fit_svm(self, X, y):
        self.device_ = self._resolve_device()
        self.classes_, encoded = np.unique(y, return_inverse=True)
        self.n_features_in_ = X.shape[1]
        if self.gamma == 'scale':
            variance = X.var()
            self.gamma_ = 1.0 / (X.shape[1] * variance) if variance > 0 else 1.0
        elif self.gamma == 'auto':
            self.gamma_ = 1.0 / X.shape[1]
        else:
            self.gamma_ = float(self.gamma)

        signed_y = 2.0 * encoded - 1.0
        self.alpha_ = np.zeros(len(X))
        gradient = -np.ones(len(X))
        eps = 1e-8 * min(1.0, self.C)
        features = None
        if self.device_ == 'cuda':
            try:
                features = self._gpu_features(X)
            except MemoryError:
                if self.device != 'auto':
                    raise
                warnings.warn('Memori GPU tidak cukup; training beralih ke CPU.', RuntimeWarning)
                self.device_ = 'cpu'

        steps_per_pass = (len(X) + 63) // 64
        self.n_iter_ = 0
        for iteration in range(self.max_iter * steps_per_pass):
            # Dua himpunan kandidat memberi batas bawah dan atas bias pada kondisi KKT.
            values = -signed_y * gradient
            up = np.flatnonzero(((signed_y > 0) & (self.alpha_ < self.C - eps))
                                | ((signed_y < 0) & (self.alpha_ > eps)))
            low = np.flatnonzero(((signed_y > 0) & (self.alpha_ > eps))
                                 | ((signed_y < 0) & (self.alpha_ < self.C - eps)))
            if values[up].max() - values[low].min() <= 2.0 * self.tol:
                break
            working = np.unique(np.concatenate((up[np.argsort(values[up])[-32:]],
                                                low[np.argsort(values[low])[:32]])))
            if features is None:
                kernel = self._kernel(X, X[working])
            else:
                import torch
                try:
                    with torch.inference_mode():
                        kernel = self._kernel(features, features[working], xp=torch).cpu().numpy()
                except torch.cuda.OutOfMemoryError as error:
                    if self.device != 'auto':
                        raise MemoryError('VRAM tidak cukup untuk blok kernel SVM.') from error
                    warnings.warn('Memori GPU tidak cukup; training beralih ke CPU.', RuntimeWarning)
                    features = None
                    self.device_ = 'cpu'
                    kernel = self._kernel(X, X[working])
            columns = signed_y[:, None] * kernel * signed_y[working]
            block = columns[working]
            old = self.alpha_[working].copy()
            labels = signed_y[working]
            linear = gradient[working] - block @ old

            # SciPy menangani optimasi QP beserta batas alpha dan kendala jumlah alpha*y = 0.
            result = minimize(lambda a: (0.5 * a @ block @ a + linear @ a, block @ a + linear),
                              old, jac=True, method='SLSQP', bounds=[(0.0, self.C)] * len(old),
                              constraints={'type': 'eq', 'fun': lambda a: labels @ (a - old),
                                           'jac': lambda a: labels},
                              options={'ftol': min(1e-9, self.tol ** 2 * 1e-3), 'maxiter': 1000})
            if not result.success:
                raise RuntimeError(f'Solver QP SVM gagal: {result.message}')
            self.alpha_[working] = result.x
            gradient += columns @ (result.x - old)
            self.n_iter_ = iteration // steps_per_pass + 1

        values = -signed_y * gradient
        up = ((signed_y > 0) & (self.alpha_ < self.C - eps)) | ((signed_y < 0) & (self.alpha_ > eps))
        low = ((signed_y > 0) & (self.alpha_ > eps)) | ((signed_y < 0) & (self.alpha_ < self.C - eps))
        self.intercept_ = float((values[up].max() + values[low].min()) / 2.0)
        self.support_ = np.flatnonzero(self.alpha_ > eps)
        self.support_vectors_ = X[self.support_].copy()
        self.dual_coef_ = self.alpha_[self.support_] * signed_y[self.support_]
        self.coef_ = self.dual_coef_ @ self.support_vectors_ if self.kernel == 'linear' else None
        residual = signed_y * self.decision_function(X) - 1.0
        self.converged_ = not (((residual < -self.tol) & (self.alpha_ < self.C - eps))
                              | ((residual > self.tol) & (self.alpha_ > eps))).any()
        if not self.converged_:
            warnings.warn('SVM berhenti sebelum seluruh kondisi KKT memenuhi tol. '
                          'Periksa scaling atau naikkan max_iter.', RuntimeWarning, stacklevel=2)
        return self

    @staticmethod
    def _gpu_features(X, torch_device='cuda'):
        import torch
        try:
            return torch.as_tensor(X, dtype=torch.float64, device=torch_device)
        except torch.cuda.OutOfMemoryError as error:
            raise MemoryError('VRAM tidak cukup untuk data SVM.') from error

    # PyTorch opsional: mode CPU dan prediksi model tersimpan tidak mengimpornya.
    def _resolve_device(self):
        device = getattr(self, 'device', 'cpu')
        if device not in ('cpu', 'cuda', 'auto'):
            raise ValueError("device harus 'cpu', 'cuda', atau 'auto'.")
        if device == 'cpu':
            return 'cpu'
        try:
            import torch
        except ImportError as error:
            if device == 'auto':
                return 'cpu'
            raise RuntimeError('device=cuda memerlukan PyTorch dengan dukungan CUDA.') from error
        if torch.cuda.is_available():
            return 'cuda'
        if device == 'auto':
            return 'cpu'
        raise RuntimeError('CUDA tidak tersedia. Aktifkan GPU dan gunakan PyTorch CUDA, atau device=cpu.')

    # Hitung kemiripan seluruh pasangan baris pada X dan Y.
    def _kernel(self, X, Y, xp=np):
        with np.errstate(over='ignore', invalid='ignore'):
            product = X @ Y.T
            if self.kernel == 'poly':
                result = (self.gamma_ * product + self.coef0) ** self.degree
            elif self.kernel in ('gaussian', 'rbf'):
                distance = (X ** 2).sum(axis=1)[:, None] + (Y ** 2).sum(axis=1) - 2.0 * product
                result = xp.exp(-self.gamma_ * distance.clip(0.0, None))
            else:
                result = product
        if not xp.isfinite(result).all():
            raise ValueError('Perhitungan kernel tidak finite, periksa scaling dan parameter kernel.')
        return result

    # Gunakan 20 persen dari tiap kelas untuk kalibrasi model sementara.
    def _calibrate(self, X, y):
        positive = y == self.classes_[1]
        counts = np.bincount(positive.astype(int), minlength=2)
        if counts.min() < 2:
            self.probA_ = 0.0
            self.probB_ = float(np.log((counts[0] + 1.0) / (counts[1] + 1.0)))
            warnings.warn('Sampel tiap kelas terlalu sedikit untuk holdout kalibrasi. '
                          'Probabilitas memakai prior kelas.', RuntimeWarning, stacklevel=2)
            return

        rng = np.random.default_rng(self.seed)
        holdout = []
        for label in self.classes_:
            indices = rng.permutation(np.flatnonzero(y == label))
            size = max(1, len(indices) // 5)
            holdout.extend(indices[:size])
        holdout = np.asarray(holdout)
        training = np.ones(len(X), dtype=bool)
        training[holdout] = False

        calibration_model = SVM(C=self.C, tol=self.tol, max_iter=self.max_iter,
                                seed=self.seed, kernel=self.kernel, degree=self.degree,
                                gamma=self.gamma_, coef0=self.coef0, device=self.device_)
        calibration_model._fit_svm(X[training], y[training])
        scores = calibration_model.decision_function(X[holdout])
        self._fit_sigmoid(scores, positive[holdout])

    # Pelajari sigmoid P(kelas kedua) = 1 / (1 + exp(A * score + B)).
    def _fit_sigmoid(self, scores, positive):
        n_positive = int(positive.sum())
        n_negative = len(positive) - n_positive
        target = np.where(positive, (n_positive + 1.0) / (n_positive + 2.0),
                          1.0 / (n_negative + 2.0))
        center = scores.mean()
        scale = scores.std()
        if scale == 0:
            scale = 1.0
        design = np.column_stack(((scores - center) / scale, np.ones(len(scores))))
        params = np.array([0.0, np.log((n_negative + 1.0) / (n_positive + 1.0))])

        def objective(params):
            logits = design @ params
            probability = np.exp(-np.logaddexp(0.0, logits))
            loss = np.sum(np.logaddexp(0.0, -logits) + target * logits)
            return loss, design.T @ (target - probability)

        result = minimize(objective, params, jac=True, method='BFGS',
                          options={'gtol': 1e-5, 'maxiter': 100})
        if not result.success and np.max(np.abs(result.jac)) > 1e-5:
            warnings.warn(f'Kalibrasi sigmoid belum konvergen: {result.message}', RuntimeWarning)
        params = result.x
        self.probA_ = float(params[0] / scale)
        self.probB_ = float(params[1] - self.probA_ * center)

    # Pastikan fitur berupa matriks numerik tanpa NaN atau infinity.
    @staticmethod
    def _validate_X(X):
        X = np.asarray(X, dtype=float)
        if X.ndim != 2 or X.shape[1] == 0 or not np.isfinite(X).all():
            raise ValueError('X harus berupa matriks numerik finite dengan minimal satu fitur.')
        return X

    # Kernel linear memakai w.x + b, kernel lainnya memakai kontribusi support vector.
    def decision_function(self, X):
        if not hasattr(self, 'support_'):
            raise RuntimeError('SVM harus di-fit sebelum melakukan prediksi.')
        X = self._validate_X(X)
        if X.shape[1] != self.n_features_in_:
            raise ValueError('Jumlah fitur harus sama dengan data train.')
        if getattr(self, 'kernel', 'linear') == 'linear':
            return X @ self.coef_ + self.intercept_
        scores = np.empty(len(X))
        for start in range(0, len(X), 256):
            kernel = self._kernel(X[start:start + 256], self.support_vectors_)
            scores[start:start + 256] = kernel @ self.dual_coef_ + self.intercept_
        return scores

    # Probabilitas kelas kedua dalam array satu dimensi untuk pipeline validasi tim.
    def predict_proba(self, X):
        if not hasattr(self, 'probA_'):
            raise RuntimeError('SVM harus di-fit dengan kalibrasi sebelum menghitung probabilitas.')
        logits = self.probA_ * self.decision_function(X) + self.probB_
        return np.exp(-np.logaddexp(0.0, logits))

    # Gunakan tanda score secara default, atau threshold probabilitas jika diberikan.
    def predict(self, X, threshold=None):
        if threshold is not None:
            if not np.isfinite(threshold) or not 0 <= threshold <= 1:
                raise ValueError('threshold harus berada dalam rentang 0 sampai 1.')
            probability = self.predict_proba(X)
            return self.classes_[(probability >= threshold).astype(int)]
        scores = self.decision_function(X)
        return self.classes_[(scores >= 0).astype(int)]

### Visualisasi SVM

Matplotlib menampilkan pemisah, margin, support vector, dan hasil evaluasi.

In [ ]:
def show_plot(fig, path):
    from IPython.display import display

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=130, bbox_inches='tight')
    display(fig)
    plt.close(fig)

def boundary_plot(X, y, models, names):
    lower, upper = X.min(axis=0) - 0.5, X.max(axis=0) + 0.5
    xx, yy = np.meshgrid(np.linspace(lower[0], upper[0], 160),
                         np.linspace(lower[1], upper[1], 160))
    grid = np.column_stack((xx.ravel(), yy.ravel()))
    fig, axes = plt.subplots(1, len(models), figsize=(13, 4), sharex=True, sharey=True, squeeze=False)
    for ax, (kernel, model) in zip(axes[0], models.items()):
        score = model.decision_function(grid).reshape(xx.shape)
        ax.contourf(xx, yy, (score >= 0).astype(int), levels=[-0.5, 0.5, 1.5],
                    colors=['tab:blue', 'tab:orange'], alpha=0.12)
        ax.contour(xx, yy, score, levels=[-1, 0, 1], colors='black',
                   linestyles=['--', '-', '--'], linewidths=[1, 1.5, 1])
        for label, color, marker, text in [(0, 'tab:blue', 'o', 'Tidak batal'),
                                           (1, 'tab:orange', '^', 'Batal')]:
            mask = y == label
            ax.scatter(X[mask, 0], X[mask, 1], color=color, marker=marker,
                       s=16, alpha=0.65, label=text)
        sv = model.support_vectors_
        ax.scatter(sv[:, 0], sv[:, 1], s=55, facecolors='none',
                   edgecolors='black', linewidths=0.7, label='Support vector')
        ax.set(title=f'{names[kernel]} ({len(sv)} SV)', xlabel='Jeda kedatangan (z-score)')
    axes[0, 0].set_ylabel('Tarif kamar (z-score)')
    handles, labels = axes[0, 0].get_legend_handles_labels()
    handles += [Line2D([], [], color='black'), Line2D([], [], color='black', linestyle='--')]
    labels += ['Pemisah f=0', 'Margin f=±1']
    fig.legend(handles, labels, loc='lower center', ncol=5)
    fig.suptitle(f'Pemisah, margin, dan support vector SVM pada {len(X)} baris train')
    fig.tight_layout(rect=[0, 0.12, 1, 0.93])
    return fig

def pso_plot(history, comparison, dev_size, holdout_size):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
    axes[0].plot(range(1, len(history) + 1), history, 'o-', label='Fitness terbaik')
    axes[0].axhline(comparison.iloc[0]['oof_macro_f1'], color='gray', linestyle='--', label='Baseline')
    axes[0].set(title=f'Pencarian PSO pada {dev_size} baris', xlabel='Generasi', ylabel='Macro F1 OOF')
    axes[0].legend()
    comparison[['oof_macro_f1', 'holdout_macro_f1']].plot.bar(ax=axes[1], rot=0)
    axes[1].set(title='Baseline dan hasil PSO', xlabel='', ylabel='Macro F1', ylim=(0, 1))
    axes[1].legend(['OOF seleksi', f'Holdout ({holdout_size} baris)'])
    return fig

def kernel_plot(results, details, names, rows):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
    table = results.pivot(index='kernel', columns='implementation', values='oof_macro_f1')
    table = table.reindex(list(names)).rename(index=names)
    table.rename(columns={'from_scratch': 'From scratch', 'sklearn': 'scikit-learn'}).plot.bar(ax=axes[0], rot=0)
    axes[0].set(title=f'OOF pada {rows:,} baris', xlabel='Kernel', ylabel='Macro F1', ylim=(0, 1))
    for kernel, label in names.items():
        part = details[(details['kernel'] == kernel) & (details['implementation'] == 'from_scratch')]
        axes[1].plot(part['fold'], part['macro_f1'], marker='o', label=label)
    axes[1].set(title='Skor from scratch tiap fold', xlabel='Fold', ylabel='Macro F1')
    axes[1].legend()
    return fig

### Utilitas Data dan Validasi

Cleaning, metrik, dan validasi silang didefinisikan di notebook ini.

In [ ]:
TARGET = 'is_canceled'

ID_COL = 'id'

CAT_COLS = ['meal_package', 'room_category', 'booking_channel']

def clean_train(df):
    feature_cols = [c for c in df.columns if c != ID_COL]
    return df.drop_duplicates(subset=feature_cols).reset_index(drop=True)

def mark_invalid(df):
    df = df.copy()
    df.loc[df['num_adults'] == 0, 'num_adults'] = np.nan
    return df

def add_features(df):
    df = df.copy()
    df['total_nights'] = df['weekend_nights'] + df['weekday_nights']
    df['total_guests'] = df['num_adults'] + df['num_children']
    df['is_free'] = (df['nightly_rate_idr'] == 0).astype(float)
    df.loc[df['nightly_rate_idr'].isna(), 'is_free'] = np.nan
    df['rate_per_guest'] = df['nightly_rate_idr'] / df['total_guests'].replace(0, np.nan)

    arrival = pd.to_datetime(
        dict(year=df['arrival_year'], month=df['arrival_month'], day=df['arrival_day']),
        errors='coerce',
    )
    df['arrival_weekday'] = arrival.dt.weekday.astype(float)
    df['arrival_dayofyear'] = arrival.dt.dayofyear.astype(float)
    df['arrival_index'] = ((arrival - pd.Timestamp('2023-01-01')).dt.days).astype(float)
    return df

def save_model(bundle, path):
    # Simpan state agar pickle tidak bergantung pada modul model di luar notebook.
    bundle = dict(bundle)
    for key in ['model', 'preprocessor']:
        obj = bundle.get(key)
        if obj is not None:
            bundle[key + '_state'] = vars(obj).copy()
            bundle[key + '_class'] = type(obj).__name__
            del bundle[key]
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('wb') as file:
        pickle.dump(bundle, file)


def load_model(path):
    with Path(path).open('rb') as file:
        bundle = pickle.load(file)
    for key, default in [('model', 'SVM'), ('preprocessor', 'SVMPreprocessor')]:
        if key + '_state' in bundle:
            cls = globals().get(bundle.get(key + '_class', default))
            if cls is None:
                raise RuntimeError('Jalankan sel definisi kelas sebelum load model.')
            obj = cls.__new__(cls)
            obj.__dict__.update(bundle[key + '_state'])
            bundle[key] = obj
    return bundle


def confusion_matrix(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    return np.array([[tn, fp], [fn, tp]])

def f1_per_class(y_true, y_pred):
    (tn, fp), (fn, tp) = confusion_matrix(y_true, y_pred)
    f1_pos = 2 * tp / (2 * tp + fp + fn) if tp + fp + fn > 0 else 0.0
    f1_neg = 2 * tn / (2 * tn + fn + fp) if tn + fn + fp > 0 else 0.0
    return np.array([f1_neg, f1_pos])

def macro_f1(y_true, y_pred):
    return float(f1_per_class(y_true, y_pred).mean())

def accuracy(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))

def best_threshold(y_true, proba, grid=None):
    if grid is None:
        grid = np.round(np.arange(0.20, 0.81, 0.01), 2)
    scores = np.array([macro_f1(y_true, (proba >= t).astype(int)) for t in grid])
    i = int(np.argmax(scores))
    return float(grid[i]), float(scores[i])

def stratified_kfold(y, n_splits=5, seed=42):
    y = np.asarray(y)
    rng = np.random.RandomState(seed)
    fold_of = np.empty(len(y), dtype=int)
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        fold_of[idx] = np.arange(len(idx)) % n_splits
    return [(np.where(fold_of != k)[0], np.where(fold_of == k)[0]) for k in range(n_splits)]

def cross_val_proba(make_model, make_prep, df, y, folds):
    y = np.asarray(y)
    oof = np.zeros(len(y))
    for tr_idx, va_idx in folds:
        prep = make_prep()
        X_tr = prep.fit_transform(df.iloc[tr_idx])
        X_va = prep.transform(df.iloc[va_idx])
        model = make_model()
        model.fit(X_tr, y[tr_idx])
        oof[va_idx] = model.predict_proba(X_va)
    return oof

def fold_scores(y, proba, folds, threshold):
    y = np.asarray(y)
    pred = (proba >= threshold).astype(int)
    return np.array([macro_f1(y[va], pred[va]) for _, va in folds])

### Implementasi Preprocessing

In [ ]:
class BasePreprocessor:

    # feature_engineering: pakai add_features atau tidak; rare_min_count: batas kategori langka.
    def __init__(self, feature_engineering=True, rare_min_count=20, drop_cols=None):
        self.feature_engineering = feature_engineering
        self.rare_min_count = rare_min_count
        self.drop_cols = list(drop_cols or [])

    # Pelajari kolom dan kategori dari data train, lalu statistik khusus model.
    def fit(self, df):
        df = self._base(df)
        self.cat_cols_ = [c for c in CAT_COLS if c in df.columns]
        self.num_cols_ = [c for c in df.columns if c not in self.cat_cols_]
        self.keep_cats_ = {}
        for c in self.cat_cols_:
            counts = df[c].value_counts()
            self.keep_cats_[c] = sorted(counts[counts >= self.rare_min_count].index)
        self._fit_extra(self._group_rare(df))
        return self

    # Terapkan transformasi yang sudah dipelajari ke data validasi atau test.
    def transform(self, df):
        return self._transform_extra(self._group_rare(self._base(df)))

    # Gabungan fit lalu transform.
    def fit_transform(self, df):
        return self.fit(df).transform(df)

    # Statistik khusus model, default tidak ada.
    def _fit_extra(self, df):
        pass

    # Transformasi khusus model, default data dikembalikan apa adanya.
    def _transform_extra(self, df):
        return df

    # Tandai nilai janggal, tambah fitur turunan (opsional), buang kolom id dan target.
    def _base(self, df):
        df = mark_invalid(df)
        if self.feature_engineering:
            df = add_features(df)
        drop = [ID_COL, TARGET] + self.drop_cols
        return df.drop(columns=[c for c in drop if c in df.columns])

    # Ganti kategori yang tidak ada di daftar train menjadi 'Other'.
    def _group_rare(self, df):
        df = df.copy()
        for c in self.cat_cols_:
            known = df[c].isin(self.keep_cats_[c]) | df[c].isna()
            df.loc[~known, c] = 'Other'
        return df

class SVMPreprocessor(BasePreprocessor):
    # Pelajari statistik dari train untuk transform data validasi dan test.
    def _fit_extra(self, df):
        # Pelajari nilai imputasi, dengan fallback untuk kolom yang seluruhnya kosong.
        self.medians_ = df[self.num_cols_].median().fillna(0.0)
        self.modes_ = {c: df[c].mode().get(0, 'Other') for c in self.cat_cols_}

        # Simpan indikator missing dan susunan kolom hasil encoding.
        self.missing_cols_ = [c for c in df.columns if df[c].isna().any()]
        encoded = self._encode(df)
        self.columns_ = list(encoded.columns)

        # Simpan statistik scaling, std nol atau tidak tersedia diganti satu.
        self.means_ = encoded.mean()
        self.stds_ = encoded.std().fillna(1.0).replace(0, 1.0)

    # Pertahankan susunan fitur serta statistik yang dipelajari dari train.
    def _transform_extra(self, df):
        encoded = self._encode(df).reindex(columns=self.columns_, fill_value=0.0)
        return (encoded - self.means_) / self.stds_

    def _encode(self, df):
        out = pd.DataFrame(index=df.index)
        for c in self.missing_cols_:
            out[f'{c}_missing'] = df[c].isna().astype(float)
        out[self.num_cols_] = df[self.num_cols_].fillna(self.medians_)
        for c in self.cat_cols_:
            filled = df[c].fillna(self.modes_[c])
            for cat in dict.fromkeys(self.keep_cats_[c] + ['Other']):
                out[f'{c}_{cat}'] = (filled == cat).astype(float)
        return out.astype(float)

## Model dan kernel


$$\min_\alpha \frac12\alpha^TQ\alpha-\sum_i\alpha_i,\quad
Q_{ij}=y_i y_jK(x_i,x_j),\quad 0\le\alpha_i\le C,\quad\sum_i\alpha_i y_i=0.$$

| Kernel | Fungsi |
| --- | --- |
| Linear | $K(x,z)=x^Tz$ |
| Polynomial | $K(x,z)=(\gamma x^Tz+c_0)^d$ |
| Gaussian | $K(x,z)=\exp(-\gamma\lVert x-z\rVert^2)$ |

Optimasi kuadratik diselesaikan menggunakan SLSQP dalam blok yang masing-masing
memuat paling banyak 64 alpha. Kelas `SVM` menghitung kernel, bias, support vector,
prediksi, dan pemeriksaan kondisi KKT.

Fungsi keputusan $f(x)=\sum_{i\in SV}\alpha_i y_iK(x_i,x)+b$ dikalibrasi dengan
sigmoid untuk memperoleh probabilitas kelas. Parameter kalibrasi dipelajari
menggunakan BFGS. Implementasi `SVC` menjadi pembanding pada tahap evaluasi.

In [ ]:
# Tampilkan implementasi kernel yang benar-benar dipakai oleh model.
display(Code(inspect.getsource(SVM._kernel), language='python'))

In [ ]:
# Baca ID sebagai teks dan hapus duplikat fitur/target; ID tidak menjadi fitur.
# SVMPreprocessor menangani nilai tidak valid, tujuh fitur turunan, imputasi,
# indikator missing, kategori langka, one-hot, dan standardisasi.
# Statistik preprocessing dipelajari hanya dari train pada setiap fold.
train_raw = pd.read_csv(DATA_DIR / 'train.csv', dtype={'id': str})
train = clean_train(train_raw)
assert set(train[TARGET].unique()) == {0, 1}
# Ambil sampel stratified hanya jika ukuran evaluasi dibatasi.
if CV_ROWS is None or CV_ROWS >= len(train):
    cv_train = train.copy()
else:
    selected, _ = train_test_split(np.arange(len(train)), train_size=CV_ROWS,
                                    stratify=train[TARGET], random_state=SEED)
    cv_train = train.iloc[selected].reset_index(drop=True)
y_cv = cv_train[TARGET].to_numpy()
# Pastikan setiap baris menjadi validasi tepat sekali dan tidak masuk train fold yang sama.
cv_folds = stratified_kfold(y_cv, n_splits=N_FOLDS, seed=SEED)
assert np.array_equal(np.sort(np.concatenate([va for _, va in cv_folds])), np.arange(len(y_cv)))
assert all(not np.intersect1d(tr, va).size for tr, va in cv_folds)
print('Train mentah:', len(train_raw), '| train bersih:', len(train), '| evaluasi:', len(cv_train))
print('Jumlah fitur mentah:', len(train.columns) - 2, '| fold:', N_FOLDS, '| seed:', SEED)
display(pd.DataFrame(KERNEL_PARAMS).T)

## Evaluasi ketiga kernel

Ketiga kernel dievaluasi dengan preprocessing, parameter kernel, dan lima fold luar
yang sama. Prediksi out-of-fold (OOF) digunakan untuk menghitung Macro F1 dan
memilih threshold.

Kalibrasi probabilitas pada implementasi from scratch menggunakan holdout internal
sebesar 20%. Pembanding scikit-learn menggunakan lima fold internal. Perbedaan
skema kalibrasi ini perlu diperhitungkan saat membandingkan skor kedua implementasi.

In [ ]:
# Samakan keluaran probabilitas scikit-learn dengan API SVM from scratch.
class SklearnWrapper:
    def __init__(self, params):
        reference_params = dict(params)
        # Gaussian pada API kita memakai identifier 'rbf' di scikit-learn.
        if reference_params['kernel'] == 'gaussian':
            reference_params['kernel'] = 'rbf'
        self.model = CalibratedClassifierCV(
            SVC(**reference_params, tol=COMMON_PARAMS['tol'], random_state=SEED),
            method='sigmoid', ensemble=False,
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED))

    def fit(self, X, y):
        self.model.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)[:, 1]

# Evaluasi ketiga kernel pada from scratch dan scikit-learn, tanpa mengganti fold.
kernel_rows, oof_by_model, fold_details = [], {}, []
with threadpool_limits(limits=1):
    for kernel, params in KERNEL_PARAMS.items():
        for implementation in ['from_scratch', 'sklearn']:
            fitted = []
            def make_model():
                model = (SVM(**COMMON_PARAMS, **params) if implementation == 'from_scratch'
                         else SklearnWrapper(params))
                fitted.append(model)
                return model

            print(f'Mulai {KERNEL_NAMES[kernel]} / {implementation}', flush=True)
            started = time.perf_counter()
            with warnings.catch_warnings(record=True) as messages:
                warnings.simplefilter('always', RuntimeWarning)
                # Fit preprocessor baru pada train tiap fold, lalu prediksi validasinya.
                oof = cross_val_proba(make_model, SVMPreprocessor, cv_train, y_cv, cv_folds)
            seconds = time.perf_counter() - started
            assert oof.shape == y_cv.shape and np.isfinite(oof).all()
            assert ((0 <= oof) & (oof <= 1)).all()
            # Pilih threshold OOF untuk Macro F1 pada grid 0.20–0.80, langkah 0.01.
            threshold, score = best_threshold(y_cv, oof)
            pred = (oof >= threshold).astype(int)
            assert np.isclose(score, f1_score(y_cv, pred, average='macro'))
            scores = fold_scores(y_cv, oof, cv_folds, threshold)
            # Simpan konvergensi dan peringatan agar skor model bermasalah tidak lolos seleksi.
            converged = sum(bool(m.converged_) for m in fitted) if implementation == 'from_scratch' else None
            kernel_rows.append(dict(kernel=kernel, implementation=implementation,
                                    rows=len(cv_train), oof_macro_f1=score, fold_std=float(scores.std()),
                                    threshold=threshold, seconds=seconds,
                                    converged_folds=converged, warnings=len(messages)))
            oof_by_model[f'{kernel}_{implementation}'] = oof
            for fold, ((_, va), fold_score) in enumerate(zip(cv_folds, scores), 1):
                model = fitted[fold - 1]
                fold_details.append(dict(kernel=kernel, implementation=implementation, fold=fold,
                                         macro_f1=float(fold_score),
                                         converged=bool(model.converged_) if implementation == 'from_scratch' else None))
            print(f'Selesai: F1={score:.4f}, threshold={threshold:.2f}, '
                  f'waktu={seconds:.1f}s, peringatan={len(messages)}', flush=True)
            for message in sorted({str(m.message) for m in messages}):
                print(message)

kernel_results = pd.DataFrame(kernel_rows)
display(kernel_results.round(4))
display(pd.DataFrame(fold_details).round(4))
# Simpan skor dan probabilitas OOF untuk penelusuran hasil.
kernel_results.to_csv(RUN_DIR / 'kernels.csv', index=False)
np.savez_compressed(RUN_DIR / 'kernel_oof.npz', y=y_cv, **oof_by_model)

In [ ]:
# Bandingkan skor OOF dan variasi lima fold. Std menyatakan penyebaran skor, bukan confidence interval.
fig = kernel_plot(kernel_results, pd.DataFrame(fold_details), KERNEL_NAMES, len(cv_train))
show_plot(fig, FIG_DIR / 'svm_final_kernels.png')

# Pilih hanya from scratch yang semua fold-nya konvergen tanpa peringatan.
# Skor PSO memakai sampel berbeda sehingga tidak dicampur dalam pemilihan ini.
eligible = kernel_results[(kernel_results['implementation'] == 'from_scratch')
                          & (kernel_results['converged_folds'] == N_FOLDS)
                          & (kernel_results['warnings'] == 0)]
if eligible.empty:
    raise RuntimeError('Tidak ada kernel dengan seluruh fold konvergen tanpa peringatan; periksa diagnostik.')
chosen = eligible.sort_values('oof_macro_f1', ascending=False, kind='stable').iloc[0]
chosen_kernel = chosen['kernel']
chosen_threshold = float(chosen['threshold'])
display(Markdown(f"Kernel {KERNEL_NAMES[chosen_kernel]} terpilih dengan Macro F1 OOF "
                 f"sebesar {chosen['oof_macro_f1']:.4f} dan threshold {chosen_threshold:.2f}. "
                 "Skor OOF juga digunakan dalam pemilihan model, sehingga hasil ini "
                 "belum memberikan estimasi independen atas kemampuan generalisasinya."))
# Rinci jenis kesalahan dan F1 tiap kelas, bukan hanya rata-rata Macro F1.
diagnostics = []
for kernel in KERNEL_PARAMS:
    pred = (oof_by_model[f'{kernel}_from_scratch'] >= float(kernel_results.loc[
        (kernel_results['kernel'] == kernel) & (kernel_results['implementation'] == 'from_scratch'), 'threshold'].iloc[0])).astype(int)
    (tn, fp), (fn, tp) = confusion_matrix(y_cv, pred)
    f1_0, f1_1 = f1_per_class(y_cv, pred)
    diagnostics.append(dict(kernel=kernel, TN=tn, FP=fp, FN=fn, TP=tp, F1_0=f1_0, F1_1=f1_1))
display(pd.DataFrame(diagnostics).round(4))

## Visualisasi dua fitur

Setiap kernel dilatih ulang menggunakan dua fitur untuk menggambarkan batas
keputusannya. Grafik ini tidak memproyeksikan model yang dilatih pada seluruh fitur.

Garis penuh menunjukkan $f(x)=0$, sedangkan garis putus-putus menunjukkan
$f(x)=\pm1$. Lingkaran menandai support vector. Pada kernel non-linear, jarak
antarkontur dalam gambar tidak menyatakan lebar margin Euclidean di ruang asal.
Support vector pada SVM soft-margin juga dapat berada di dalam margin.

In [ ]:
# Siapkan sampel bonus terpisah dari populasi evaluasi utama.
# PSO hanya memakai data pengembangan. Holdout dievaluasi setelah pencarian selesai.
train = clean_train(pd.read_csv(DATA_DIR / 'train.csv', dtype={'id': str}))
sample_idx, _ = train_test_split(np.arange(len(train)), train_size=DEV_SIZE + HOLDOUT_SIZE,
                                 stratify=train[TARGET], random_state=SEED)
dev_idx, holdout_idx = train_test_split(sample_idx, test_size=HOLDOUT_SIZE,
                                       stratify=train.iloc[sample_idx][TARGET], random_state=SEED)
assert not np.intersect1d(dev_idx, holdout_idx).size
dev = train.iloc[dev_idx].reset_index(drop=True)
holdout = train.iloc[holdout_idx].reset_index(drop=True)
y_dev = dev[TARGET].to_numpy()
folds = stratified_kfold(y_dev, n_splits=5, seed=SEED)
assert np.array_equal(np.sort(np.concatenate([va for _, va in folds])), np.arange(DEV_SIZE))
# Fit preprocessing sekali per train fold dan pakai ulang transform yang sama untuk semua kandidat.
prepared_folds = []
for tr, va in folds:
    assert not np.intersect1d(tr, va).size
    prep = SVMPreprocessor()
    prepared_folds.append((tr, va, prep.fit_transform(dev.iloc[tr]).to_numpy(),
                           prep.transform(dev.iloc[va]).to_numpy()))
display(pd.DataFrame({'bagian': ['Pengembangan', 'Holdout'],
                      'baris': [len(dev), len(holdout)],
                      'proporsi_batal': [y_dev.mean(), holdout[TARGET].mean()]}).round(4))

In [ ]:
# Gunakan waktu tunggu dan tarif kamar untuk memperlihatkan bentuk batas keputusan.
visual_idx, _ = train_test_split(np.arange(len(dev)), train_size=VISUAL_SIZE,
                                 stratify=y_dev, random_state=SEED)
features_2d = ['days_before_arrival', 'nightly_rate_idr']
raw_2d = dev.iloc[visual_idx][features_2d]
# Hitung median dan statistik standardisasi dari sampel visualisasi. Tarif nol tetap disertakan.
filled_2d = raw_2d.fillna(raw_2d.median())
X_2d = ((filled_2d - filled_2d.mean()) / filled_2d.std(ddof=0).replace(0, 1)).to_numpy()
y_2d = y_dev[visual_idx]
assert X_2d.shape == (VISUAL_SIZE, 2) and np.isfinite(X_2d).all()
# Latih tiga model 2D pada sampel yang sama agar bentuk batasnya dapat dibandingkan.
visual_models = {}
with threadpool_limits(limits=1):
    for kernel in ['linear', 'poly', 'gaussian']:
        model = SVM(kernel=kernel, C=1, degree=3, coef0=1, gamma='scale',
                    tol=0.01, max_iter=1000, seed=SEED, device='auto').fit(X_2d, y_2d)
        assert model.converged_
        visual_models[kernel] = model
    fig = boundary_plot(X_2d, y_2d, visual_models,
                        {'linear': 'Linear', 'poly': 'Polynomial', 'gaussian': 'Gaussian'})
visual_results = [dict(kernel=k, support_vectors=len(m.support_), converged=bool(m.converged_))
                  for k, m in visual_models.items()]
show_plot(fig, FIG_DIR / 'svm_boundaries.png')

## Optimasi PSO

Setiap partikel mewakili kandidat hyperparameter $x=(\log_{10}C,\log_{10}g)$ dengan
$\gamma=g/(d\,\mathrm{Var}(X_{train}))$. Posisi kandidat diperbarui berdasarkan posisi terbaik
partikel $p_i$ dan posisi terbaik seluruh swarm $g^*$:

$$v_i\leftarrow wv_i+c_1r_1(p_i-x_i)+c_2r_2(g^*-x_i),\qquad x_i\leftarrow x_i+v_i.$$

Fitness kandidat dihitung dari Macro F1 OOF setelah pemilihan threshold. PSO
mencari hyperparameter, sementara SLSQP melatih SVM pada setiap kandidat. Nilai
fitness terbaik dipertahankan selama pencarian, tetapi peningkatan pada holdout
dan pencapaian optimum global tetap tidak terjamin.

Implementasi PSO mengacu pada referensi [1] dan [2]. Penerapannya untuk tuning SVM
mengacu pada referensi [3]. Eksperimen ini tidak menerapkan feature selection yang
dibahas dalam penelitian tersebut.

In [ ]:
# PSO dengan inertia weight: Kennedy & Eberhart (1995), doi:10.1109/ICNN.1995.488968;
# Shi & Eberhart (1998), doi:10.1109/ICEC.1998.699146.
def particle_swarm(objective, bounds, initial, particles=8, generations=4, seed=42):
    bounds = np.asarray(bounds, dtype=float)
    initial = np.asarray(initial, dtype=float)
    if (bounds.shape != (len(initial), 2) or not np.isfinite(bounds).all()
            or not np.isfinite(initial).all() or np.any(bounds[:, 0] >= bounds[:, 1])
            or np.any(initial < bounds[:, 0]) or np.any(initial > bounds[:, 1])
            or particles < 2 or generations < 1):
        raise ValueError('Batas, posisi awal, dan anggaran PSO tidak valid.')
    rng = np.random.default_rng(seed)
    lower, upper = bounds.T
    positions = rng.uniform(lower, upper, size=(particles, len(initial)))
    # Sertakan baseline sebagai satu posisi awal, lalu simpan pengalaman terbaik tiap partikel.
    positions[0] = initial
    velocity = np.zeros_like(positions)
    personal = positions.copy()
    personal_scores = np.full(particles, -np.inf)
    trace = []
    for generation in range(generations):
        scores = np.array([objective(position.copy()) for position in positions])
        if not np.isfinite(scores).all():
            raise ValueError('Fitness PSO harus finite.')
        # Perbarui personal best dan global best berdasarkan fitness yang benar-benar dievaluasi.
        improved = scores > personal_scores
        personal[improved], personal_scores[improved] = positions[improved], scores[improved]
        best = int(np.argmax(personal_scores))
        global_best = personal[best].copy()
        trace.append(float(personal_scores[best]))
        if generation + 1 < generations:
            # Inertia 0.7 mempertahankan sebagian arah. Bobot 1.5 menyeimbangkan pengalaman pribadi dan swarm.
            velocity = (0.7 * velocity
                        + 1.5 * rng.random(positions.shape) * (personal - positions)
                        + 1.5 * rng.random(positions.shape) * (global_best - positions))
            # Batasi langkah dan posisi agar pencarian tetap berada dalam rentang hyperparameter.
            velocity = np.clip(velocity, -0.25 * (upper - lower), 0.25 * (upper - lower))
            positions = np.clip(positions + velocity, lower, upper)
    return global_best, np.asarray(trace)

# Uji independen pada fungsi cekung dengan optimum diketahui di (0.3, -0.4).
def check_swarm():
    visited = []
    target = np.array([0.3, -0.4])
    def objective(point):
        assert ((point >= -1) & (point <= 1)).all()
        visited.append(point)
        return -float(np.sum((point - target) ** 2))
    args = dict(bounds=[[-1, 1], [-1, 1]], initial=[0, 0], particles=8, generations=30)
    best, history = particle_swarm(objective, **args)
    assert len(visited) == 240 and np.all(np.diff(history) >= 0)
    np.testing.assert_allclose(best, target, atol=0.02)
    repeated, _ = particle_swarm(objective, **args)
    np.testing.assert_array_equal(best, repeated)
check_swarm()
print('PSO: batas posisi, anggaran, perbaikan fitness, optimum uji, dan seed lolos.')

In [ ]:
# Nilai tiap kandidat PSO dengan fold dan preprocessing yang sama.
# Solver yang gagal atau belum konvergen menghentikan evaluasi kandidat.
evaluations = []
def evaluate(position):
    # Skala log menjadikan perpindahan partikel sebagai perubahan rasio C dan gamma.
    C, gamma_factor = 10.0 ** np.asarray(position)
    oof = np.full(DEV_SIZE, np.nan)
    started = time.perf_counter()
    for tr, va, X_tr, X_va in prepared_folds:
        # Faktor 1 setara gamma='scale' pada train fold. Holdout tidak ikut menghitung varians.
        gamma = gamma_factor / (X_tr.shape[1] * X_tr.var())
        with warnings.catch_warnings(record=True) as messages:
            warnings.simplefilter('always', RuntimeWarning)
            model = SVM(**BASE_PARAMS, C=C, gamma=gamma).fit(X_tr, y_dev[tr])
        assert model.converged_ and not messages, [str(m.message) for m in messages]
        oof[va] = model.predict_proba(X_va)
    assert np.isfinite(oof).all() and ((oof >= 0) & (oof <= 1)).all()
    threshold, score = best_threshold(y_dev, oof)
    assert np.isclose(score, f1_score(y_dev, oof >= threshold, average='macro'))
    evaluations.append(dict(evaluation=len(evaluations) + 1, C=float(C),
                            gamma_factor=float(gamma_factor), threshold=threshold,
                            oof_macro_f1=score,
                            fold_std=float(fold_scores(y_dev, oof, folds, threshold).std()),
                            seconds=time.perf_counter() - started))
    print(f'Kandidat {len(evaluations):02d}/{PARTICLES * GENERATIONS}: '
          f'C={C:.3g}, faktor gamma={gamma_factor:.3g}, OOF={score:.4f}', flush=True)
    return score

# Rentang C 0.01–10 dan faktor gamma 0.05–5 mencakup penalti serta lebar kernel di sekitar baseline.
started = time.perf_counter()
with threadpool_limits(limits=1):
    best_position, history = particle_swarm(
        evaluate, bounds=[[-2, 1], [np.log10(0.05), np.log10(5)]], initial=[0, 0],
        particles=PARTICLES, generations=GENERATIONS, seed=SEED)
search_seconds = time.perf_counter() - started
evaluations = pd.DataFrame(evaluations)
# Ambil baseline awal dan kandidat pemenang sesuai posisi terbaik PSO.
baseline = evaluations.iloc[0].to_dict()
matches = np.isclose(evaluations['C'], 10.0 ** best_position[0]) & np.isclose(
    evaluations['gamma_factor'], 10.0 ** best_position[1])
best = evaluations.loc[matches].iloc[0].to_dict()
np.testing.assert_allclose(10.0 ** best_position, [best['C'], best['gamma_factor']])
assert len(evaluations) == PARTICLES * GENERATIONS
assert history[-1] >= baseline['oof_macro_f1'] and np.all(np.diff(history) >= 0)
display(evaluations.nlargest(5, 'oof_macro_f1').round(4))
print(f'Pencarian selesai: {search_seconds:.1f} detik.')

In [ ]:
# Latih ulang baseline dan pemenang pada data pengembangan, lalu nilai holdout.
# Threshold tetap berasal dari OOF. Hasil holdout tidak digunakan untuk memilih kandidat baru.
prep_final = SVMPreprocessor()
X_dev = prep_final.fit_transform(dev).to_numpy()
X_holdout = prep_final.transform(holdout).to_numpy()
y_holdout = holdout[TARGET].to_numpy()
comparison_rows = []
with threadpool_limits(limits=1):
    for name, candidate in [('Baseline Gaussian', baseline), ('Gaussian + PSO', best)]:
        gamma = candidate['gamma_factor'] / (X_dev.shape[1] * X_dev.var())
        started = time.perf_counter()
        with warnings.catch_warnings(record=True) as messages:
            warnings.simplefilter('always', RuntimeWarning)
            model = SVM(**BASE_PARAMS, C=candidate['C'], gamma=gamma).fit(X_dev, y_dev)
        assert model.converged_ and not messages, [str(m.message) for m in messages]
        pred = model.predict(X_holdout, threshold=candidate['threshold'])
        score = macro_f1(y_holdout, pred)
        assert np.isclose(score, f1_score(y_holdout, pred, average='macro'))
        comparison_rows.append(dict(model=name, C=candidate['C'], gamma_factor=candidate['gamma_factor'],
                                    threshold=candidate['threshold'], oof_macro_f1=candidate['oof_macro_f1'],
                                    fold_std=candidate['fold_std'], holdout_macro_f1=score,
                                    fit_seconds=time.perf_counter() - started))
comparison = pd.DataFrame(comparison_rows).set_index('model')
display(comparison.round(4))

In [ ]:
# Plot riwayat fitness serta perbandingan skor seleksi dengan holdout.
fig = pso_plot(history, comparison, DEV_SIZE, HOLDOUT_SIZE)
show_plot(fig, FIG_DIR / 'svm_pso.png')

In [ ]:
# Jelaskan hasil yang teramati, termasuk jika peningkatan OOF tidak bertahan di holdout.
oof_gain = comparison.iloc[1]['oof_macro_f1'] - comparison.iloc[0]['oof_macro_f1']
holdout_gain = comparison.iloc[1]['holdout_macro_f1'] - comparison.iloc[0]['holdout_macro_f1']
holdout_result = 'meningkat' if holdout_gain > 0 else ('tetap' if holdout_gain == 0 else 'menurun')
display(Markdown(
    f"PSO memilih C={best['C']:.4g}, faktor gamma={best['gamma_factor']:.4g}, "
    f"dan threshold={best['threshold']:.2f}. Macro F1 OOF berubah dari "
    f"{baseline['oof_macro_f1']:.4f} menjadi {best['oof_macro_f1']:.4f}, "
    f"dengan selisih {oof_gain:+.4f}. Skor ini digunakan untuk memilih kandidat. "
    f"\n\nPada holdout terpisah, Macro F1 {holdout_result} dari "
    f"**{comparison.iloc[0]['holdout_macro_f1']:.4f}** menjadi "
    f"**{comparison.iloc[1]['holdout_macro_f1']:.4f}** (selisih {holdout_gain:+.4f}). "
    f"Pencarian memakai {len(evaluations)} evaluasi dengan waktu {search_seconds:.1f} detik. "
    "\n\nEksperimen dengan satu seed dan holdout berukuran kecil belum cukup untuk "
    "menyimpulkan bahwa konfigurasi terpilih lebih baik secara umum atau meningkatkan "
    "skor Kaggle. Jumlah evaluasi yang terbatas juga belum membuktikan optimum global."
))
# Simpan kandidat, metrik, hash data, dan ID pembagian agar eksperimen dapat ditelusuri.
evaluations.to_csv(RUN_DIR / 'evaluations.csv', index=False)
comparison.to_csv(RUN_DIR / 'comparison.csv')
summary = dict(seed=SEED, dev_rows=DEV_SIZE, holdout_rows=HOLDOUT_SIZE,
               particles=PARTICLES, generations=GENERATIONS, search_seconds=search_seconds,
               train_sha256=hashlib.sha256((DATA_DIR / 'train.csv').read_bytes()).hexdigest(),
               dev_ids=dev['id'].tolist(), holdout_ids=holdout['id'].tolist(),
               visualizations=visual_results, history=history.tolist(),
               comparison=comparison.reset_index().to_dict(orient='records'))
(RUN_DIR / 'results.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Gambar:', (FIG_DIR / 'svm_boundaries.png').relative_to(ROOT),
      'dan', (FIG_DIR / 'svm_pso.png').relative_to(ROOT))

In [ ]:
# Buat artefak akhir dari kernel pemenang OOF utama, bukan skor sampel PSO.
# Latih model akhir dengan seluruh data latih bersih, lalu transformasikan data test.
test_raw = pd.read_csv(DATA_DIR / 'test.csv', dtype={'id': str})
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv', dtype={'id': str})
assert list(sample.columns) == ['id', TARGET]
assert test_raw['id'].notna().all() and test_raw['id'].is_unique
assert sample['id'].is_unique and test_raw['id'].equals(sample['id'])

# Pelajari statistik dari data latih dan gunakan statistik tersebut untuk data test.
prep_final = SVMPreprocessor()
X_train = prep_final.fit_transform(train)
X_test = prep_final.transform(test_raw)
assert TARGET not in X_train.columns and 'id' not in X_train.columns
assert X_train.columns.equals(X_test.columns)
started = time.perf_counter()
with threadpool_limits(limits=1):
    with warnings.catch_warnings(record=True) as messages:
        warnings.simplefilter('always', RuntimeWarning)
        model_final = SVM(**COMMON_PARAMS, **KERNEL_PARAMS[chosen_kernel]).fit(X_train, train[TARGET].to_numpy())
    if not model_final.converged_ or messages:
        raise RuntimeError('Training akhir belum bersih: ' + '; '.join(str(m.message) for m in messages))
    probability_before = model_final.predict_proba(X_test)
    pred_before = model_final.predict(X_test, threshold=chosen_threshold)
    # Simpan preprocessor, model, threshold, dan ukuran data evaluasi dalam satu bundle.
    save_model({'preprocessor': prep_final, 'model': model_final, 'threshold': chosen_threshold,
                'cv_rows': len(cv_train), 'train_rows': len(train)}, MODEL_PATH)
    # Verifikasi hasil load identik dengan model sebelum disimpan.
    bundle = load_model(MODEL_PATH)
    loaded_X = bundle['preprocessor'].transform(test_raw)
    np.testing.assert_array_equal(bundle['model'].predict_proba(loaded_X), probability_before)
    pred = bundle['model'].predict(loaded_X, threshold=bundle['threshold'])
    np.testing.assert_array_equal(pred, pred_before)
assert pred.shape == (len(test_raw),) and np.isin(pred, [0, 1]).all()
# Pertahankan kolom, jumlah baris, dan urutan ID sesuai contoh submission.
submission = pd.DataFrame({'id': test_raw['id'], TARGET: pred})
assert submission.shape == sample.shape and submission.columns.equals(sample.columns)
assert submission['id'].equals(sample['id'])
SUB_PATH.parent.mkdir(parents=True, exist_ok=True)
# Ekspor lokal dan baca kembali untuk memeriksa isi CSV.
submission.to_csv(SUB_PATH, index=False)
pd.testing.assert_frame_equal(submission, pd.read_csv(SUB_PATH, dtype={'id': str}))
print(f'Training akhir selesai: {time.perf_counter() - started:.1f} detik, '
      f'{len(train)} baris, {KERNEL_NAMES[chosen_kernel]}, konvergen={model_final.converged_}')
print('Model:', MODEL_PATH)
print('CSV:', SUB_PATH)
display(submission.head())

## Referensi

- Riwayat eksperimen: [`docs/EXPERIMENTS.md`](../../../docs/EXPERIMENTS.md).
- Solver QP: [SciPy SLSQP](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-slsqp.html).
- Kalibrasi sigmoid: [SciPy BFGS](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-bfgs.html).

1. J. Kennedy dan R. Eberhart, “Particle Swarm Optimization,” ICNN, 1995,
   hlm. 1942–1948. [DOI: 10.1109/ICNN.1995.488968](https://doi.org/10.1109/ICNN.1995.488968).
2. Y. Shi dan R. Eberhart, “A Modified Particle Swarm Optimizer,” IEEE ICEC, 1998,
   hlm. 69–73. [DOI: 10.1109/ICEC.1998.699146](https://doi.org/10.1109/ICEC.1998.699146).
3. S.-W. Lin, K.-C. Ying, S.-C. Chen, dan Z.-J. Lee,
   “Particle swarm optimization for parameter determination and feature selection
   of support vector machines,” Expert Systems with Applications, vol. 35, 2008,
   hlm. 1817–1824. [DOI: 10.1016/j.eswa.2007.08.088](https://doi.org/10.1016/j.eswa.2007.08.088).